In [1]:
import pandas as pd
import re
import numpy as np



In [2]:
# Path to the folder where you downloaded the files
folder_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/association_overall_direct'

# Read all parquet parts at once
df = pd.read_parquet(folder_path, engine='pyarrow')

# Show the first few rows
print(df.head(3))

      diseaseId         targetId     score  evidenceCount
0  DOID_0050890  ENSG00000001084  0.031799              4
1  DOID_0050890  ENSG00000004142  0.002217              1
2  DOID_0050890  ENSG00000004478  0.002217              1


In [16]:

ot_disease = pd.read_parquet('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/disease/disease.parquet')
ot_disease.head(3)


,id,code,name,description,dbXRefs,parents,synonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology
0,DOID_0050890,http://purl.obolibrary.org/obo/DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...,"[MESH:D000080874, MONDO:0000510, UMLS:C5191670...","[MONDO_0019052, MONDO_0021179, MONDO_0024237]",{'hasExactSynonym': ['alpha Synucleinopathies'...,[],[],"[EFO_0006792, EFO_1001050]","[MONDO_0024237, EFO_0005772, EFO_0000618, MOND...","[EFO_0000618, OTAR_0000018, OTAR_0000020]","[MONDO_0000211, MONDO_0016418, MONDO_0014889, ...","{'isTherapeuticArea': False, 'leaf': False, 's..."
1,DOID_10113,http://purl.obolibrary.org/obo/DOID_10113,trypanosomiasis,Infection with protozoa of the genus trypanosoma.,"[UMLS:C0041227, MONDO:0000940, ICD10CM:B56, Me...",[MONDO_0002428],{'hasExactSynonym': ['Trypanosoma caused disea...,[],[],"[MONDO_0001444, EFO_0005225, EFO_0008559]","[MONDO_0002428, EFO_0001067, EFO_0005741]",[EFO_0005741],"[EFO_0005225, EFO_0005529, EFO_0008559, MONDO_...","{'isTherapeuticArea': False, 'leaf': False, 's..."
2,DOID_10718,http://purl.obolibrary.org/obo/DOID_10718,giardiasis,An infection of the small intestine caused by ...,"[MeSH:D005873, ICD9CM:007.1, MESH:D005873, MON...","[MONDO_0002428, EFO_0009561]","{'hasExactSynonym': ['giardiasis', 'beaver fea...",[],[],[],"[EFO_0010282, EFO_0009431, EFO_0009561, EFO_00...","[EFO_0010282, EFO_0005741]",[],"{'isTherapeuticArea': False, 'leaf': False, 's..."


In [40]:
# ICD10 dictionary
icd_dict = {
    'Certain infectious and parasitic diseases': ['A00', 'B99'],
    'Neoplasms': ['C00', 'D48'],
    'Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism': ['D50', 'D89'],
    'Endocrine, nutritional and metabolic diseases': ['E00', 'E90'],
    'Mental and behavioural disorders': ['F00', 'F99'],
    'Diseases of the nervous system': ['G00', 'G99'],
    'Diseases of the eye and adnexa': ['H00', 'H59'],
    'Diseases of the ear and mastoid process': ['H60', 'H95'],
    'Diseases of the circulatory system': ['I00', 'I99'],
    'Diseases of the respiratory system': ['J00', 'J99'],
    'Diseases of the digestive system': ['K00', 'K93'],
    'Diseases of the skin and subcutaneous tissue': ['L00', 'L99'],
    'Diseases of the musculoskeletal system and connective tissue': ['M00', 'M99'],
    'Diseases of the genitourinary system': ['N00', 'N99']
}

# Preprocess ICD ranges for fast lookup
icd_ranges = []
for category, (start, end) in icd_dict.items():
    icd_ranges.append((start, end))

# Function to check if an ICD10 code is within the defined ranges
def icd_in_range(icd_code):
    match = re.match(r'^ICD10:([A-Z]\d{2})$', icd_code.strip())
    if match:
        code = match.group(1)
        for start, end in icd_ranges:
            if start[0] == end[0] == code[0]:  # Same starting letter
                if int(start[1:]) <= int(code[1:]) <= int(end[1:]):
                    return True
    return False

# Function to get the first matching ICD10 code in a row
def get_icd_code(row):
    if isinstance(row, np.ndarray):
        for ref in row:
            if isinstance(ref, str) and icd_in_range(ref):
                return ref.strip()  # Return the first matching ICD10 code
    return None

filtered_df = ot_disease[ot_disease['dbXRefs'].apply(get_icd_code).notnull()].copy()

filtered_df['icd'] = filtered_df['dbXRefs'].apply(get_icd_code)



In [60]:
filtered_dga = df[df['diseaseId'].isin(filtered_df['id'].unique())]
filtered_dga = filtered_dga.rename(columns={'diseaseId': 'id'})

In [62]:
merged_df = pd.merge(filtered_dga, filtered_df[['id', 'icd']], on='id', how='outer')

unique_rows = merged_df.drop_duplicates(subset=['icd', 'targetId'])


In [65]:
unique_rows

,id,targetId,score,evidenceCount,icd
0,EFO_0000217,ENSG00000001617,0.007392,1.0,ICD10:K29
1,EFO_0000217,ENSG00000001626,0.001478,1.0,ICD10:K29
2,EFO_0000217,ENSG00000003400,0.001478,1.0,ICD10:K29
3,EFO_0000217,ENSG00000003436,0.003696,1.0,ICD10:K29
4,EFO_0000217,ENSG00000003756,0.007392,1.0,ICD10:K29
...,...,...,...,...,...
295908,Orphanet_98757,ENSG00000277586,0.090816,19.0,ICD10:G11
295909,Orphanet_98757,ENSG00000282728,0.005813,13.0,ICD10:G11
295911,Orphanet_98757,ENSG00000292332,0.007392,1.0,ICD10:G11
295912,Orphanet_98757,ENSG00000292357,0.002957,1.0,ICD10:G11


In [67]:
# Function to check if a row contains an ICD10 code starting with 'ICD10:C'
def has_icd10_c(row):
    if isinstance(row, np.ndarray):
        return any(isinstance(ref, str) and ref.startswith('ICD10:C50') for ref in row)
    return False

ot_disease[ot_disease['dbXRefs'].apply(has_icd10_c)]

,id,code,name,description,dbXRefs,parents,synonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology
25327,EFO_0003869,http://www.ebi.ac.uk/efo/EFO_0003869,breast neoplasm,A benign or malignant neoplasm of the breast p...,"[MONDO:0021100, MeSH:D001943, ONCOTREE:BREAST,...","[MONDO_0021350, EFO_0009483, EFO_0010285]","{'hasExactSynonym': ['Breast Neoplasms', 'Mamm...",[],[],"[MONDO_0000620, MONDO_0002482, MONDO_0002483, ...","[MONDO_0021350, EFO_0010285, MONDO_0045024, MO...","[EFO_0010285, MONDO_0045024, OTAR_0000017]","[MONDO_0003959, EFO_0009649, EFO_1000254, MOND...","{'isTherapeuticArea': False, 'leaf': False, 's..."
35196,EFO_1000040,http://www.ebi.ac.uk/efo/EFO_1000040,metaplastic breast carcinoma,A group of invasive breast carcinomas characte...,"[ICD10:C50.1, ICD10:C50.6, MEDGEN:277360, ICD1...",[EFO_1000307],{'hasExactSynonym': ['metaplastic breast carci...,[EFO_1000374],[NCIt:C5164],[EFO_1000053],"[MONDO_0007254, MONDO_0004992, EFO_0009483, MO...","[EFO_0010285, OTAR_0000017, MONDO_0045024]","[EFO_1000053, MONDO_0004231, EFO_1001969, MOND...","{'isTherapeuticArea': False, 'leaf': False, 's..."
37143,Orphanet_145,http://www.orpha.net/ORDO/Orphanet_145,Hereditary breast and ovarian cancer syndrome,An autosomal dominant inherited syndrome cause...,"[SCTID:718220008, UMLS:C0677776, MESH:D061325,...","[MONDO_0002229, MONDO_0008170, Orphanet_227535]","{'hasExactSynonym': ['breast-ovarian cancer, f...",[EFO_0002611],[DOID:5683],[],"[Orphanet_227535, OTAR_0000018, MONDO_0021350,...","[EFO_0010285, OTAR_0000017, EFO_0001379, OTAR_...",[],"{'isTherapeuticArea': False, 'leaf': False, 's..."
37537,Orphanet_227535,http://www.orpha.net/ORDO/Orphanet_227535,Hereditary breast cancer,Breast carcinoma that has developed in relativ...,"[ICD10:C50.3, ICD10:C50.5, NCIT:C4503, Orphane...","[MONDO_0002149, EFO_0000305, Orphanet_183734]","{'hasExactSynonym': ['familial breast cancer',...",[],[],[Orphanet_145],"[MONDO_0004992, EFO_0000616, MONDO_0023370, EF...","[OTAR_0000017, MONDO_0045024, EFO_0010285, OTA...",[Orphanet_145],"{'isTherapeuticArea': False, 'leaf': False, 's..."


In [7]:
disgenet = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/disgent_with_time.csv')
disgenet.head(3)

,disease_id,omim,hpo,disease_name,gene_id,score,first_pub_year,last_pub_year,ei,dsi,dpi,uniprot_id,string_id,ori_annotation
0,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,ERBB2,1.0,2004.0,2007.0,0.917,0.298,0.957,P04626,9606.ENSP00000269571,True
1,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,PIK3CA,1.0,2004.0,2023.0,0.978,0.275,0.957,P42336,9606.ENSP00000263967,True
2,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,TP53,1.0,2011.0,2023.0,0.911,0.256,0.957,P04637,9606.ENSP00000269305,True


In [70]:
disgenet_ids = []
for id in disgenet['disease_id'].unique().tolist():
    disgenet_ids.append(str(id.split('_')[0]+':'+id.split('_')[1]))

In [ ]:
def get_disgenet_ids(row):
    """
    Return a list of matching IDs from disgenet_ids in the row.
    """
    if isinstance(row, (np.ndarray, list)):
        for ref in row :
            if isinstance(ref, str) and ref in disgenet_ids:
                return ref
    return []

# Apply the function to extract matching IDs
filtered_df = ot_disease.copy()  # Make a safe copy to avoid warnings
filtered_df['icd'] = filtered_df['dbXRefs'].apply(get_disgenet_ids)

# Optional: Keep only rows where at least one ID matched
filtered_df = filtered_df[filtered_df['icd'].apply(lambda x: len(x) > 0)]


In [ ]:
filtered_dga = df[df['diseaseId'].isin(filtered_df['id'].unique())]
filtered_dga = filtered_dga.rename(columns={'diseaseId': 'id'})
merged_df = pd.merge(filtered_dga, filtered_df[['id', 'icd']], on='id', how='outer')

unique_rows = merged_df.drop_duplicates(subset=['icd', 'targetId'])

In [ ]:
unique_rows['disease_id'] = unique_rows['icd'].str.split(':').str.join('_')
unique_rows.head(3)

/tmp/ipykernel_3565115/1273831005.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unique_rows['disease_id'] = unique_rows['icd'].str.split(':').str.join('_')


,id,targetId,score,evidenceCount,icd,disease_id
0,EFO_0000183,ENSG00000000460,0.004435,1.0,ICD10:C81,ICD10_C81
1,EFO_0000183,ENSG00000000938,0.046197,2.0,ICD10:C81,ICD10_C81
2,EFO_0000183,ENSG00000001461,0.045266,1.0,ICD10:C81,ICD10_C81


In [98]:
import mygene

In [99]:
mg = mygene.MyGeneInfo()
# Query mygene for UniProt and Entrez gene ID mappings
results = mg.querymany(
    unique_rows['targetId'].unique().tolist(),
    scopes='ensembl.gene',
    fields='ensembl.protein',
    species='human'
)
results_df = pd.DataFrame(results)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('ENSG00000280018', 2)]
5 input query terms found no hit:	['ENSG00000168078', 'ENSG00000281376', 'ENSG00000189144', 'ENSG00000310561', 'nan']


In [101]:
results_df = results_df[~results_df['ensembl'].isna()]

In [103]:
map_df = pd.DataFrame({
    'ensg': results_df['query'],
    'ensp': results_df['ensembl'].apply(lambda x: x.get('protein') if isinstance(x, dict) else None)
})

# Step 2: Explode the list of proteins to one per row
map_df = map_df.explode('ensp').reset_index(drop=True)

In [108]:
unique_rows = unique_rows.rename(columns={'targetId': 'ensg'})

In [113]:
# Step 1: Merge the dataframes
mapped_ot_dga = pd.merge(unique_rows, map_df, on='ensg', how='left')


In [115]:
mapped_ot_dga['string_id'] = '9606.'+mapped_ot_dga['ensp']

In [124]:
mapped_ot_dga[['score', 'disease_id', 'string_id']].to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_dgas.csv',index=False)

In [116]:
# Assuming merged_df and disgent_df are already loaded

# Step 1: Create a 'merge key' in both dataframes
mapped_ot_dga['merge_key'] = mapped_ot_dga['string_id'] + '|' + mapped_ot_dga['disease_id']
disgenet['merge_key'] = disgenet['string_id'] + '|' + disgenet['disease_id']

# Step 2: Keep only rows in merged_df where merge_key is not in disgent_df
non_overlap_df = mapped_ot_dga[~mapped_ot_dga['merge_key'].isin(disgenet['merge_key'])].drop(columns='merge_key')


In [120]:
non_overlap_df.columns

Index(['id', 'ensg', 'score', 'evidenceCount', 'icd', 'disease_id', 'ensp',
       'string_id'],
      dtype='object')

In [122]:
non_overlap_df[['score', 'disease_id', 'string_id']].to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/unique_ot_dgas.csv',index=False)

# deal with missing disease

In [18]:
unique_ot_dgas = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/unique_ot_dgas.csv')
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/disgent_with_time.csv')
ppi_feature = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_700_emb.csv')
all_df = all_df[all_df['string_id'].isin(ppi_feature['string_id'])]

In [6]:
unique_ot_dgas['disease_id'].unique().shape

(44,)

In [19]:
time = 2017
selected_diseases = []
for disease_id in all_df['disease_id'].unique():
    sub_df = all_df[all_df['disease_id']==disease_id]
    if len(sub_df) < 15:
        continue
    else:
        # print(type(time),type(sub_df['first_pub_year'].max()))
        if sub_df['first_pub_year'].max() > time and sub_df['first_pub_year'].min() <= time and len(sub_df[sub_df['first_pub_year']<time]) >=5:
            selected_diseases.append(disease_id)

In [20]:
len(selected_diseases)

49

In [22]:
query_disease = list(set(selected_diseases)-set(unique_ot_dgas['disease_id'].unique().tolist()))

In [23]:
diseaseid_map = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ICD2EFO_map.tsv',sep = '\t')
diseaseid_map = diseaseid_map[['MAPPED_TERM_URI','ICD10_CODE/SELF_REPORTED_TRAIT_FIELD_CODE']]
diseaseid_map.rename(columns={"MAPPED_TERM_URI": "EFO"}, inplace=True)
diseaseid_map.rename(columns={"ICD10_CODE/SELF_REPORTED_TRAIT_FIELD_CODE": "disease_id"}, inplace=True)
diseaseid_map.dropna(inplace=True)

In [28]:
len(query_disease)

17

In [27]:
diseaseid_map[diseaseid_map['disease_id'].isin([items[-3: ]for items in query_disease])]

,EFO,disease_id
85,EFO_0000249,G30
215,EFO_0000516,H40
217,EFO_0000692,F20
225,EFO_0001359,E10
236,EFO_0003821,G43
239,EFO_0003885,G35
363,Orphanet_1572,D83
365,Orphanet_232,D57
367,Orphanet_399,G10
368,Orphanet_586,E84
